## 清洗数据
去除重复行和空行，并输出覆盖率

In [1]:
import pandas as pd
import os

def deduplicate_and_dropna_csv(
    input_file,
    target_cols,
    dropna_cols=None,
    output_file=None,
    keep='first',
    encoding='utf-8'
):
    """
    对CSV文件的指定列去重 + 统计特定列去空后的行数，输出详细统计结果并保存去重文件
    
    参数:
        input_file: 输入CSV文件路径（必填）
        target_cols: 需去重的列名（单列传字符串，多列传列表，如 'CID' 或 ['CID', 'SMILES']）
        dropna_cols: 需统计去空行数的列名（默认与target_cols一致，可单独指定，如 'SMILES' 或 ['CID', 'name']）
        output_file: 输出去重后的CSV文件路径（默认：输入文件名_deduplicated.csv）
        keep: 去重保留规则（'first'=保留首次出现，'last'=保留末次出现，False=删除所有重复行）
        encoding: 文件编码（默认utf-8，支持gbk、latin1等）
    """
    # 1. 基础校验：文件是否存在
    if not os.path.exists(input_file):
        print(f"❌ 错误：输入文件 '{input_file}' 不存在")
        return

    # 2. 读取CSV文件（容错处理）
    try:
        df = pd.read_csv(input_file, encoding=encoding)
        print(f"✅ 成功读取文件：{input_file}")
        original_total = len(df)
        print(f"\n📊 基础统计（原始文件）：")
        print(f"   原始总行数：{original_total}")
    except Exception as e:
        print(f"❌ 读取文件失败：{str(e)}")
        return

    # 3. 统一处理列名参数（转为列表，方便后续操作）
    if isinstance(target_cols, str):
        target_cols = [target_cols]
    if dropna_cols is None:
        dropna_cols = target_cols  # 默认去空统计列 = 去重列
    elif isinstance(dropna_cols, str):
        dropna_cols = [dropna_cols]

    # 4. 校验指定列是否存在
    missing_target = [col for col in target_cols if col not in df.columns]
    missing_dropna = [col for col in dropna_cols if col not in df.columns]
    if missing_target or missing_dropna:
        print(f"❌ 错误：文件中缺少指定列")
        if missing_target:
            print(f"   去重列缺失：{', '.join(missing_target)}")
        if missing_dropna:
            print(f"   去空统计列缺失：{', '.join(missing_dropna)}")
        print(f"   文件可用列：{df.columns.tolist()}")
        return

    # 5. 统计特定列去空后的行数（核心新增功能）
    # 逻辑：统计 "所有指定去空列均非空" 的行数
    df_dropna = df.dropna(subset=dropna_cols)
    dropna_valid_rows = len(df_dropna)
    dropna_empty_rows = original_total - dropna_valid_rows

    print(f"\n📊 去空统计（针对列：{', '.join(dropna_cols)}）：")
    print(f"   所有指定列均非空的行数：{dropna_valid_rows}")
    print(f"   至少一列为空的行数：{dropna_empty_rows}")
    print(f"   非空率：{dropna_valid_rows/original_total*100:.2f}%")

    # 6. 按指定列去重 + 统计
    df_deduplicated = df.drop_duplicates(subset=target_cols, keep=keep, ignore_index=True)
    dedup_total = len(df_deduplicated)
    removed_dup_rows = original_total - dedup_total

    print(f"\n📊 去重统计（针对列：{', '.join(target_cols)}）：")
    print(f"   去重后总行数：{dedup_total}")
    print(f"   删除的重复行数：{removed_dup_rows}")
    print(f"   去重保留规则：{'保留首次出现' if keep == 'first' else '保留末次出现' if keep == 'last' else '删除所有重复行'}")

    # 7. 统计去重后 + 去空的行数（附加指标，便于用户评估数据质量）
    df_dedup_dropna = df_deduplicated.dropna(subset=dropna_cols)
    dedup_dropna_rows = len(df_dedup_dropna)
    print(f"\n📊 联合统计（去重 + 去空）：")
    print(f"   去重后且所有指定列均非空的行数：{dedup_dropna_rows}")
    print(f"   最终有效数据占原始数据比例：{dedup_dropna_rows/original_total*100:.2f}%")

    # 8. 保存去空后的文件
    if output_file is None:
        base, ext = os.path.splitext(input_file)
        output_file = f"{base}_cleaned{ext}"

    try:
        df_dedup_dropna.to_csv(output_file, index=False, encoding=encoding)
        print(f"\n✅ 去重后的文件已保存至：{output_file}")
    except Exception as e:
        print(f"❌ 保存文件失败：{str(e)}")


In [2]:
# ==================== 示例：修改以下参数 ====================
input_csv = "output/smiles_annotation关联结果.csv"  # 输入CSV文件路径
target_columns = "CID"  # 需去重的列（单列示例）
# target_columns = ["CID", "SMILES"]  # 多列去重示例（组合去重）

dropna_columns = ["Description"]  # 需统计去空行数的列（可单独指定）
# dropna_columns = None  # 默认与去重列一致

output_csv = None  # 默认自动生成输出路径，可手动指定（如 "final_deduplicated.csv"）
# ===========================================================

# 执行功能
deduplicate_and_dropna_csv(
    input_file=input_csv,
    target_cols=target_columns,
    dropna_cols=dropna_columns,
    keep='first',  # 保留第一次出现的重复行
    encoding='utf-8'  # 若文件编码为gbk，改为 encoding='gbk'
)

❌ 错误：输入文件 'output/smiles_annotation关联结果.csv' 不存在


In [7]:
# ==================== 示例：修改以下参数 ====================
input_csv = "output/smiles_annotation关联结果 (1).csv"  # 输入CSV文件路径
target_columns = "CID"  # 需去重的列（单列示例）
# target_columns = ["CID", "SMILES"]  # 多列去重示例（组合去重）

dropna_columns = ["Description"]  # 需统计去空行数的列（可单独指定）
# dropna_columns = None  # 默认与去重列一致

output_csv = None  # 默认自动生成输出路径，可手动指定（如 "final_deduplicated.csv"）
# ===========================================================

# 执行功能
deduplicate_and_dropna_csv(
    input_file=input_csv,
    target_cols=target_columns,
    dropna_cols=dropna_columns,
    keep='first',  # 保留第一次出现的重复行
    encoding='utf-8'  # 若文件编码为gbk，改为 encoding='gbk'
)

✅ 成功读取文件：output/smiles_annotation关联结果 (1).csv

📊 基础统计（原始文件）：
   原始总行数：16142

📊 去空统计（针对列：Description）：
   所有指定列均非空的行数：12122
   至少一列为空的行数：4020
   非空率：75.10%

📊 去重统计（针对列：CID）：
   去重后总行数：15276
   删除的重复行数：866
   去重保留规则：保留首次出现

📊 联合统计（去重 + 去空）：
   去重后且所有指定列均非空的行数：11301
   最终有效数据占原始数据比例：70.01%

✅ 去重后的文件已保存至：output/smiles_annotation关联结果 (1)_cleaned.csv


## 调整格式
调整dicription

In [7]:
import re
import pandas as pd
from tqdm import tqdm
import os


def clean_up_description(description):
    """清理描述文本，提取第一句话并修正格式问题"""
    description = description + " "  # 便于句尾处理

    # # 移除冗余前缀
    # if description.startswith("Pure "):
    #     description = description.replace("Pure ", "")

    # 强化冗余前缀移除：支持多空格、大小写兼容（如 Pure、pure、PURE）
    # 用正则匹配开头的冗余前缀，彻底移除
    description = re.sub(r'^Pure\s+', '', description, flags=re.IGNORECASE)
    
    # 修正特定拼写错误
    if description.startswith("Mercurycombines"):
        description = description.replace("Mercurycombines", "Mercury combines")

    # 处理特殊化合物名称的格式问题（确保主谓结构）
    special_cases = {
        '17-Hydroxy-6-methylpregna-3,6-diene-3,20-dione. ': '17-Hydroxy-6-methylpregna-3,6-diene-3,20-dione is ',
        '5-Thymidylic acid. ': '5-Thymidylic acid. is ',
        "5'-S-(3-Amino-3-carboxypropyl)-5'-thioadenosine. ": "5'-S-(3-Amino-3-carboxypropyl)-5'-thioadenosine. is ",
        "Guanosine 5'-(trihydrogen diphosphate), monoanhydride with phosphorothioic acid. ": "Guanosine 5'-(trihydrogen diphosphate), monoanhydride with phosphorothioic acid is ",
        "5'-Uridylic acid. ": "5'-Uridylic acid is ",
        "5'-Adenylic acid, ": "5'-Adenylic acid is ",
        "Uridine 5'-(tetrahydrogen triphosphate). ": "Uridine 5'-(tetrahydrogen triphosphate). is ",
        "Inosine 5'-Monophosphate. ": "Inosine 5'-Monophosphate. is ",
        "Pivaloyloxymethyl butyrate (AN-9), ": "Pivaloyloxymethyl butyrate (AN-9) is ",
        "4-Amino-5-cyano-7-(D-ribofuranosyl)-7H- pyrrolo(2,3-d)pyrimidine. ": "4-Amino-5-cyano-7-(D-ribofuranosyl)-7H- pyrrolo(2,3-d)pyrimidine is ",
        "Cardamonin (also known as Dihydroxymethoxychalcone), ": "Cardamonin (also known as Dihydroxymethoxychalcone) is ",
        "Lithium has been used to treat ": "Lithium is ",
        "4,4'-Methylenebis ": "4,4'-Methylenebis is ",
        "2,3,7,8-Tetrachlorodibenzo-p-dioxin": "2,3,7,8-Tetrachlorodibenzo-p-dioxin is ",
        "Exposure to 2,4,5-trichlorophenol ": "2,4,5-Trichlorophenol exposure "
    }
    for old, new in special_cases.items():
        description = description.replace(old, new)

    # 提取第一句话（以句点+空格+大写字母为分隔标志）
    L = len(description)
    start_index = 0
    # 处理特殊前缀
    if description.startswith('C.I. '):
        start_index = len('C.I. ')
    elif description.startswith('Nectriapyrone. D '):
        start_index = len('Nectriapyrone. D ')
    elif description.startswith('Salmonella enterica sv. Minnesota LPS core oligosaccharide'):
        start_index = len('Salmonella enterica sv. Minnesota LPS core oligosaccharide')

    # 查找第一句话的结束位置
    index = 0
    for index in range(start_index, L - 1):
        if index < L - 2 and description[index] == '.' and description[index + 1] == ' ' and 'A' <= description[index + 2] <= 'Z':
            break
    first_sentence = description[:index + 1]
    return first_sentence


def detect_and_replace(sentence):
    """检测描述中的动词，替换为主语（This molecule/These molecules）"""
    target_words = {
        'is': 'This molecule',
        'was': 'This molecule',
        'appears': 'This molecule',
        'occurs': 'This molecule',
        'stands for': 'This molecule',
        'belongs to': 'This molecule',
        'exists': 'This molecule',
        'has been used in trials': 'This molecule',
        'has been investigated': 'This molecule',
        'has many uses': 'This molecule',
        'are': 'These molecules',
        'were': 'These molecules'
    }

    # 匹配动词（忽略大小写）
    match = re.search(r'\b(is|was|appears|occurs|stands for|belongs to|exists|has been used in trials|has been investigated|has many uses|are|were)\b', sentence, flags=re.IGNORECASE)
    if not match:
        return None

    first_word = match.group(1).lower()
    return target_words.get(first_word, None)


def extract_name(name_raw, description):
    """从描述中提取化合物名称，并重构描述（统一主语）"""
    first_sentence = clean_up_description(description)
    splitter = '  --  --  '  # 临时分隔符
    replaced_words = detect_and_replace(first_sentence)

    if replaced_words is None:
        return None, None, None  # 无法提取有效信息

    # 分割句子以提取名称（基于动词）
    for verb in [' is ', ' are ', ' was ', ' were ', ' appears ', ' occurs ', ' stands for ', 
                 ' belongs to ', ' exists ', ' has been used in trials ', ' has been investigated ', ' has many uses ']:
        first_sentence = first_sentence.replace(verb, splitter)

    # 提取名称
    if splitter in first_sentence:
        extracted_name = first_sentence.split(splitter, 1)[0]
    elif first_sentence.startswith(name_raw):
        extracted_name = name_raw
    elif name_raw in first_sentence:
        extracted_name = name_raw
    else:
        extracted_name = None  # 名称提取失败

    # 重构描述（替换名称为主语）
    if extracted_name is not None:
        extracted_description = description.replace(extracted_name, replaced_words)
    else:
        extracted_description = description

    return extracted_name, extracted_description, first_sentence


def process_dataframe(df):
    """处理输入数据框，返回清理后的结果"""
    processed_rows = []
    for index, row in tqdm(df.iterrows(), total=len(df)):
        # 适配新表头：Name -> 原name，Description -> 原description，SMILES保持不变
        name_raw = row['Name'].strip()
        description = row['Description'].strip()
        cid = row['CID']
        smiles = row['SMILES']

        # 提取名称和规范化描述
        extracted_name, extracted_description, first_sentence = extract_name(name_raw, description)

        # 过滤无效数据
        if extracted_name is None or extracted_description is None:
            continue

        # # 确保描述以统一主语开头（验证处理结果）
        # assert extracted_description.startswith(('This molecule', 'These molecules')), f"处理失败: {extracted_description}"

        # 优化断言：允许描述前有少量空格，且更友好的报错信息
        expected_prefixes = ('This molecule', 'These molecules')
        if not extracted_description.startswith(expected_prefixes):
            # 最后尝试清理前缀后再校验
            cleaned_final = re.sub(r'^[\s\w]+\s+(This|These)', r'\1', extracted_description, flags=re.IGNORECASE)
            if cleaned_final.startswith(expected_prefixes):
                extracted_description = cleaned_final
            else:
                print(f"警告：处理失败（跳过该行）- CID:{cid}, 描述:{extracted_description[:50]}...")
                continue

        # 构建新行
        new_row = {
            'CID': cid,
            'Name': extracted_name,  # 保留大写Name
            'SMILES': smiles,
            'Description': extracted_description  # 保留大写Description
        }
        processed_rows.append(new_row)

    return pd.DataFrame(processed_rows)


In [11]:
# 表头为CID,SMILES,Name,Description
input_file = "output/smiles_annotation关联结果_cleaned(ITCM).csv"    # 你的ChEBI来源文件
output_file = os.path.join("output/processed/", "smiles_annotation关联结果_cleaned(ITCM)_processed.csv")
# -----------------------------------------------------------

# 读取输入文件（使用正确的分隔符，这里假设是制表符\t，若为逗号可改为','）
df = pd.read_csv(input_file, delimiter=',')
print(f"原始数据量: {len(df)}")

# 处理两个数据集
processed_df = process_dataframe(df)
print(f"处理后数据量: {len(processed_df)}")

# 保存结果
processed_df.to_csv(output_file, index=False, sep='\t')
print(f"已保存至 {output_file}")

原始数据量: 14902


100%|██████████| 14902/14902 [00:01<00:00, 7909.15it/s]


处理后数据量: 7275
已保存至 output/processed/smiles_annotation关联结果_cleaned(ITCM)_processed.csv
